# Taller 4 integrador de Deep Learning
## Modelos generativos: Conditional GAN y Variational Autoencoder sobre CIFAR-10

**Integrantes:**

- Juan David Tejedor Medina
- Miguel Gerardo Moreno Aveldaño

**Conjunto de datos:** CIFAR-10  

### Edición para portafolio

Trabajo académico recuperado de la especialización. Se conservan el código, las explicaciones y las atribuciones originales. Se retiraron las salidas y los metadatos de ejecución para facilitar su lectura y revisión. Las conclusiones conservadas pertenecen a la entrega original; los entrenamientos de Deep Learning no se repitieron al organizar este repositorio. Ver [procedencia y autoría](../../docs/PROCEDENCIA.md).


## 1. Planteamiento del problema

El propósito del taller es diseñar, entrenar y comparar dos modelos generativos sobre **CIFAR-10**, un conjunto de 60.000 imágenes RGB de 32 × 32 píxeles distribuidas en diez categorías: *airplane, automobile, bird, cat, deer, dog, frog, horse, ship* y *truck*.

La dificultad es mayor que en Fashion-MNIST porque CIFAR-10 contiene tres canales de color, fondos variados, cambios de iluminación, diferentes poses y objetos con texturas complejas. Por esta razón, las arquitecturas deben adaptarse para producir tensores de tamaño 32 × 32 × 3.

Se estudiarán dos enfoques:

- **Conditional GAN (cGAN):** aprende a generar una imagen a partir de ruido aleatorio y una etiqueta de clase. El Generador intenta crear imágenes convincentes y el Discriminador intenta distinguir imágenes reales de falsas, verificando además su coherencia con la clase indicada.
- **Variational Autoencoder (VAE):** aprende una distribución latente continua para comprimir, reconstruir y generar imágenes. Su pérdida combina el error de reconstrucción con una regularización probabilística.

Los objetivos experimentales son comprobar si la cGAN responde a la etiqueta solicitada, analizar la estabilidad de su entrenamiento, evaluar qué información conserva el VAE, explorar la continuidad de su espacio latente y comparar calidad visual, diversidad, reconstrucción e interpretabilidad.


### 1.1 Preparación del entorno

importamos las librerías necesarias para el manejo de datos, visualización y construcción de redes neuronales. También fijamos una semilla para reducir la variabilidad entre ejecuciones y definimos los nombres oficiales de las diez clases de CIFAR-10.

TensorFlow y Keras se utilizan para construir y entrenar los modelos; NumPy para operaciones numéricas; y Matplotlib para las gráficas y cuadrículas de imágenes.


In [ ]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras import backend as K
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy

print('TensorFlow version:', tf.__version__)
print('GPU disponible:', len(tf.config.list_physical_devices('GPU')) > 0)

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

CLASS_NAMES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

## 2. Exploración y preparación del dataset

Antes de construir los modelos examinamos las dimensiones, el balance de clases, el rango de los píxeles y ejemplos reales. Esta revisión permite justificar la normalización y asegurar que las capas de salida de ambos modelos sean compatibles con los datos.


### 2.1 Carga, dimensiones y rango original

cargamos CIFAR-10 directamente desde `tensorflow.keras.datasets`, separamos entrenamiento y prueba, convertimos las etiquetas de forma `(N, 1)` a vectores de forma `(N,)` y mostramos las dimensiones, el número de clases, el rango original de píxeles y el tipo de dato.


In [ ]:
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

y_train = y_train.flatten()
y_test = y_test.flatten()

print('Forma x_train:', x_train.shape)
print('Forma y_train:', y_train.shape)
print('Forma x_test:', x_test.shape)
print('Forma y_test:', y_test.shape)
print('Numero de clases:', len(np.unique(y_train)))
print('Rango original de pixeles: [{}, {}]'.format(x_train.min(), x_train.max()))
print('Tipo de dato:', x_train.dtype)

el conjunto de entrenamiento contiene **50.000 imágenes** y el de prueba **10.000**. Cada imagen tiene forma **32 × 32 × 3**, es decir, 32 píxeles de alto, 32 de ancho y tres canales RGB. Hay **10 clases**, y los píxeles originales son enteros `uint8` en el rango **[0, 255]**. Estas dimensiones coinciden con la especificación de CIFAR-10 y serán la forma de entrada y salida de las redes.


### 2.2 Distribución de imágenes por clase

contamos cuántas imágenes de entrenamiento pertenecen a cada categoría y representamos los conteos en una gráfica de barras. También imprimimos el valor exacto por clase.


In [ ]:
clases, conteos = np.unique(y_train, return_counts=True)

plt.figure(figsize=(8, 4))
plt.bar([CLASS_NAMES[c] for c in clases], conteos, color='steelblue')
plt.xticks(rotation=45)
plt.ylabel('Numero de imagenes')
plt.title('Distribucion de clases en el conjunto de entrenamiento')
plt.tight_layout()
plt.show()

for c, n in zip(clases, conteos):
    print(f'{CLASS_NAMES[c]:12s}: {n} imagenes')

las diez barras tienen la misma altura y cada clase contiene exactamente **5.000 imágenes de entrenamiento**. Por tanto, CIFAR-10 está balanceado. Esto evita que una categoría domine el aprendizaje únicamente por aparecer con mayor frecuencia; las diferencias posteriores de calidad se relacionarán principalmente con la dificultad visual de cada clase y con el comportamiento de los modelos.


### 2.3 Ejemplos de las diez categorías

seleccionamos aleatoriamente dos imágenes de cada categoría y las organizamos en una cuadrícula de diez filas. Con ello verificamos visualmente el contenido del dataset


In [ ]:
n_ejemplos = 2
fig, axes = plt.subplots(len(CLASS_NAMES), n_ejemplos, figsize=(n_ejemplos * 2, len(CLASS_NAMES) * 2))

for i, clase in enumerate(CLASS_NAMES):
    idx_clase = np.where(y_train == i)[0]
    idx_muestra = np.random.choice(idx_clase, n_ejemplos, replace=False)
    for j, idx in enumerate(idx_muestra):
        axes[i, j].imshow(x_train[idx])
        axes[i, j].axis('off')
        if j == 0:
            axes[i, j].set_ylabel(clase)
    axes[i, 0].set_title(clase, loc='left', fontsize=9)

plt.tight_layout()
plt.show()

incluso dentro de una misma categoría se observan cambios de fondo, color, pose, escala e iluminación. Además, la baja resolución hace que algunas clases visualmente cercanas, como *cat* y *dog*, sean difíciles de separar. Esta variabilidad explica por qué generar CIFAR-10 es un problema más exigente que producir siluetas en escala de grises.


### 2.4 Normalización y relación con la activación de salida

Los píxeles originales están en `[0, 255]`. Para entrenar las redes los transformamos al rango **[-1, 1]** mediante:

`x_normalizado = (x / 127.5) - 1`

Esta elección es compatible con la activación **`tanh`** utilizada en la salida del Generador de la cGAN y del Decoder del VAE, porque `tanh` también produce valores entre -1 y 1. La coherencia de rangos evita que el Discriminador identifique imágenes falsas únicamente por su escala y facilita que el Decoder compare reconstrucciones con las imágenes normalizadas.


definimos una función para normalizar y otra para regresar los valores al rango visible `[0, 255]`. Después normalizamos entrenamiento y prueba, verificamos el nuevo rango y comparamos una imagen original con su versión normalizada y posteriormente desnormalizada.


In [ ]:
def normalizar(x):
    return (x.astype('float32') / 127.5) - 1.0

def desnormalizar(x):
    return ((x + 1.0) * 127.5).astype('uint8')

x_train_norm = normalizar(x_train)
x_test_norm = normalizar(x_test)

print('Rango despues de normalizar:', x_train_norm.min(), x_train_norm.max())

# verificacion visual: la imagen desnormalizada debe verse igual a la original
idx = 0
fig, axes = plt.subplots(1, 2, figsize=(5, 3))
axes[0].imshow(x_train[idx])
axes[0].set_title('Original')
axes[0].axis('off')
axes[1].imshow(desnormalizar(x_train_norm[idx]))
axes[1].set_title('Desnormalizada')
axes[1].axis('off')
plt.show()

el rango normalizado es exactamente **[-1, 1]**. La comparación visual confirma que la función inversa recupera la apariencia original, por lo que la transformación no elimina información y es compatible con las salidas `tanh` de ambos modelos.


## 3. Desarrollo de la Conditional GAN

La cGAN aprende la relación `G(z, y) → x_falsa`, donde `z` es un vector latente aleatorio y `y` es la clase solicitada. El Discriminador recibe el par `(x, y)` y estima la probabilidad de que la imagen sea real y coherente con esa etiqueta.


### 3.1 Diseño del Generador

Se eligió un vector latente `z` de dimensión **100**, suficiente para representar variaciones de color, forma y fondo sin aumentar excesivamente el costo computacional.

La etiqueta se convierte en un vector entrenable mediante `Embedding(10, 100)` y se concatena con `z`. El vector combinado se proyecta a un tensor de **4 × 4 × 256**. Tres convoluciones transpuestas duplican progresivamente la resolución espacial: **4 × 4 → 8 × 8 → 16 × 16 → 32 × 32**. La capa final produce tres canales RGB con activación `tanh`.


#### 3.1.1 Construcción del Generador

construimos el Generador con dos entradas —ruido y etiqueta—, incorporamos la condición mediante un *embedding* y usamos capas densas, normalización por lotes y convoluciones transpuestas para obtener una imagen RGB de 32 × 32 píxeles. Finalmente imprimimos el resumen de la arquitectura.


In [ ]:
Z_DIM = 100
NUM_CLASSES = 10

def construir_generador(z_dim=Z_DIM, num_classes=NUM_CLASSES):
    entrada_z = layers.Input(shape=(z_dim,), name='z_input')
    entrada_y = layers.Input(shape=(1,), name='label_input')

    embedding_y = layers.Embedding(num_classes, z_dim)(entrada_y)
    embedding_y = layers.Flatten()(embedding_y)

    x = layers.Concatenate()([entrada_z, embedding_y])

    x = layers.Dense(4 * 4 * 256, use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)
    x = layers.Reshape((4, 4, 256))(x)

    x = layers.Conv2DTranspose(128, kernel_size=4, strides=2, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)

    x = layers.Conv2DTranspose(64, kernel_size=4, strides=2, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)

    x = layers.Conv2DTranspose(32, kernel_size=4, strides=2, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)

    salida = layers.Conv2D(3, kernel_size=3, padding='same', activation='tanh', name='generated_image')(x)

    modelo = Model([entrada_z, entrada_y], salida, name='Generador')
    return modelo

generador = construir_generador()
generador.summary()

el Generador contiene **1.526.475 parámetros**, de los cuales **1.517.835 son entrenables**. Las estadísticas internas de las capas `BatchNormalization` explican los parámetros no entrenables. El resumen confirma que la resolución aumenta en las etapas previstas hasta llegar a `(32, 32, 3)`.

#### 3.1.2 Verificación explícita de dimensiones

enviamos cuatro vectores latentes y cuatro etiquetas distintas al Generador para comprobar la forma y el rango numérico de su salida antes del entrenamiento.


In [ ]:
z_prueba = tf.random.normal((4, Z_DIM))
y_prueba = tf.constant([[0], [3], [6], [9]])

img_prueba = generador([z_prueba, y_prueba])
print('Forma de salida del Generador:', img_prueba.shape)
print('Rango de valores generados: [{:.3f}, {:.3f}]'.format(
    float(tf.reduce_min(img_prueba)), float(tf.reduce_max(img_prueba))))

para un lote de cuatro entradas se obtienen cuatro imágenes con forma **(4, 32, 32, 3)**. Los valores iniciales se encuentran cerca de cero y dentro de `[-1, 1]`, como exige `tanh`. La baja amplitud inicial es normal porque los pesos todavía no han aprendido la distribución de CIFAR-10.


### 3.2 Diseño del Discriminador

El Discriminador debe estimar `D(x, y) → P(real)`. La etiqueta se transforma mediante un `Embedding` de 1.024 valores, se reorganiza como un mapa de **32 × 32 × 1** y se concatena con la imagen RGB. Así, la red procesa un tensor de cuatro canales: tres corresponden a la imagen y uno representa la condición.

Las convoluciones reducen la resolución y extraen características de forma, textura y color. Al comparar esas características con el mapa de clase, el modelo puede aprender simultáneamente si la imagen parece real y si resulta coherente con la etiqueta suministrada. La salida sigmoide representa una probabilidad entre 0 y 1.


#### 3.2.1 Construcción del Discriminador

construimos el Discriminador condicionado, concatenamos la imagen con la representación espacial de la etiqueta y aplicamos tres bloques convolucionales con `LeakyReLU` y `Dropout`. Finalmente mostramos el resumen del modelo.


In [ ]:
def construir_discriminador(num_classes=NUM_CLASSES):
    entrada_x = layers.Input(shape=(32, 32, 3), name='image_input')
    entrada_y = layers.Input(shape=(1,), name='label_input')

    embedding_y = layers.Embedding(num_classes, 32 * 32)(entrada_y)
    embedding_y = layers.Reshape((32, 32, 1))(embedding_y)

    x = layers.Concatenate()([entrada_x, embedding_y])

    x = layers.Conv2D(64, kernel_size=4, strides=2, padding='same')(x)
    x = layers.LeakyReLU(0.2)(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Conv2D(128, kernel_size=4, strides=2, padding='same')(x)
    x = layers.LeakyReLU(0.2)(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Conv2D(256, kernel_size=4, strides=2, padding='same')(x)
    x = layers.LeakyReLU(0.2)(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Flatten()(x)
    salida = layers.Dense(1, activation='sigmoid', name='real_or_fake')(x)

    modelo = Model([entrada_x, entrada_y], salida, name='Discriminador')
    return modelo

discriminador = construir_discriminador()
discriminador.summary()

el Discriminador contiene **674.241 parámetros entrenables**. La resolución disminuye de 32 × 32 a 4 × 4 mientras aumentan los filtros, lo que permite resumir la evidencia visual antes de producir una única probabilidad.

#### 3.2.2 Verificación explícita de dimensiones

 evaluamos las cuatro imágenes de prueba generadas anteriormente junto con sus etiquetas y comprobamos que el Discriminador entregue una probabilidad por ejemplo.


In [ ]:
pred_prueba = discriminador([img_prueba, y_prueba])
print('Forma de salida del Discriminador:', pred_prueba.shape)
print('Valores de probabilidad:', pred_prueba.numpy().flatten())

la salida tiene forma **(4, 1)**. Las probabilidades iniciales son cercanas a **0,5**, comportamiento esperado en una red sin entrenar: todavía no posee evidencia para separar imágenes reales y falsas.


### 3.3 Entrenamiento adversarial

El entrenamiento alterna dos actualizaciones:

1. El Discriminador aprende con imágenes reales etiquetadas como 0,9 —*label smoothing*— e imágenes generadas etiquetadas como 0.
2. El Generador produce un nuevo lote y aprende a lograr que el Discriminador lo clasifique como real.

Se utiliza entropía cruzada binaria y optimizadores Adam independientes con tasa de aprendizaje `2e-4` y `beta_1=0.5`, configuración común para estabilizar GANs.


#### 3.3.1 Hiperparámetros, datos y funciones de pérdida

definimos lotes de 128 imágenes, 30 épocas, los dos optimizadores, las funciones de pérdida y el flujo `tf.data`. El `label smoothing` reduce la confianza excesiva del Discriminador y puede mejorar la estabilidad del juego adversarial.


In [ ]:
BATCH_SIZE = 128
EPOCHS = 30

bce = BinaryCrossentropy()

opt_generador = Adam(learning_rate=2e-4, beta_1=0.5)
opt_discriminador = Adam(learning_rate=2e-4, beta_1=0.5)

dataset_gan = tf.data.Dataset.from_tensor_slices((x_train_norm, y_train))
dataset_gan = dataset_gan.shuffle(10000).batch(BATCH_SIZE, drop_remainder=True)

def loss_discriminador(pred_real, pred_fake):
    real_smooth = tf.ones_like(pred_real) * 0.9
    fake_labels = tf.zeros_like(pred_fake)
    loss_real = bce(real_smooth, pred_real)
    loss_fake = bce(fake_labels, pred_fake)
    return loss_real + loss_fake

def loss_generador(pred_fake):
    return bce(tf.ones_like(pred_fake), pred_fake)

#### 3.3.2 Paso alternado de entrenamiento

la función `train_step` realiza en cada lote una actualización del Discriminador y luego una del Generador. `tf.function` compila esta operación como un grafo para acelerar la ejecución en GPU. La función devuelve `Loss_D` y `Loss_G` para registrar el comportamiento de ambos modelos.


In [ ]:
@tf.function
def train_step(imagenes_reales, etiquetas_reales):
    batch_size = tf.shape(imagenes_reales)[0]
    ruido = tf.random.normal((batch_size, Z_DIM))

    with tf.GradientTape() as tape_d:
        imagenes_falsas = generador([ruido, etiquetas_reales], training=True)
        pred_real = discriminador([imagenes_reales, etiquetas_reales], training=True)
        pred_fake = discriminador([imagenes_falsas, etiquetas_reales], training=True)
        loss_d = loss_discriminador(pred_real, pred_fake)

    grads_d = tape_d.gradient(loss_d, discriminador.trainable_variables)
    opt_discriminador.apply_gradients(zip(grads_d, discriminador.trainable_variables))

    ruido = tf.random.normal((batch_size, Z_DIM))
    with tf.GradientTape() as tape_g:
        imagenes_falsas = generador([ruido, etiquetas_reales], training=True)
        pred_fake = discriminador([imagenes_falsas, etiquetas_reales], training=True)
        loss_g = loss_generador(pred_fake)

    grads_g = tape_g.gradient(loss_g, generador.trainable_variables)
    opt_generador.apply_gradients(zip(grads_g, generador.trainable_variables))

    return loss_d, loss_g

el paso de entrenamiento queda preparado sin depender de funciones externas ni archivos manuales. Las imágenes fijas para evaluar la evolución se definen antes de iniciar las épocas, de modo que la comparación use exactamente los mismos `z` y `y`.

inicializamos las listas donde se almacenarán las pérdidas promedio de cada época.


In [ ]:
historial_loss_d = []
historial_loss_g = []

#### 3.3.3 Cómo interpretar las pérdidas de una GAN

A diferencia de un clasificador, en una GAN no se espera una disminución monótona simultánea de ambas pérdidas. El Generador y el Discriminador tienen objetivos opuestos, por lo que es normal observar oscilaciones: cuando uno mejora, el otro recibe un problema más difícil.

Por ello, `Loss_D` y `Loss_G` sirven para detectar desequilibrios —por ejemplo, una red que domina completamente—, pero no miden por sí solas la calidad visual. La evaluación debe combinar las curvas con imágenes obtenidas mediante entradas fijas y muestras de cada clase.


## 4. Evaluación cualitativa de la cGAN

### 4.1 Evidencia de evolución del Generador

Para aislar el efecto del aprendizaje usamos siempre los mismos diez vectores latentes y una etiqueta por clase. Guardamos las imágenes al inicio, en la época 15 y al finalizar la época 30.


#### 4.1.1 Vectores y etiquetas fijos

creamos `z_fixed` y `y_fixed` una sola vez y definimos una función de visualización que desnormaliza las imágenes, muestra una por clase y conserva los nombres correspondientes.


In [ ]:
N_FIXED = 10
z_fixed = tf.random.normal((N_FIXED, Z_DIM))
y_fixed = tf.constant([[i] for i in range(N_FIXED)])

def mostrar_grilla(imagenes, titulo, etiquetas=CLASS_NAMES):
    imagenes = (imagenes + 1.0) / 2.0
    imagenes = np.clip(imagenes, 0, 1)
    fig, axes = plt.subplots(1, len(imagenes), figsize=(len(imagenes) * 1.5, 2))
    for i, ax in enumerate(axes):
        ax.imshow(imagenes[i])
        ax.set_title(etiquetas[i], fontsize=8)
        ax.axis('off')
    fig.suptitle(titulo)
    plt.tight_layout()
    plt.show()

quedan almacenadas diez entradas invariables, una por cada clase. Esto permite atribuir los cambios visuales al entrenamiento y no a un nuevo muestreo de ruido.

#### 4.1.2 Estado inicial

generamos la primera cuadrícula antes de actualizar los pesos. Esta salida funciona como línea base del experimento.


In [ ]:
imagenes_inicio = generador([z_fixed, y_fixed], training=False).numpy()
mostrar_grilla(imagenes_inicio, 'Antes de entrenar (pesos aleatorios)')

antes del entrenamiento las salidas son casi uniformes y grises, con pequeñas variaciones sin estructura. Esto ocurre porque `tanh` recibe activaciones cercanas a cero y el Generador todavía no ha aprendido formas, colores ni diferencias entre clases.


#### 4.1.3 Ciclo completo de entrenamiento

entrenamos durante 30 épocas, calculamos las pérdidas promedio de cada época y guardamos automáticamente las imágenes fijas en la época 15 y en la época 30. El tiempo de cada época también se imprime para documentar la ejecución.


In [ ]:
imagenes_intermedias = None
epoca_intermedia = EPOCHS // 2

for epoca in range(1, EPOCHS + 1):
    t0 = time.time()
    losses_d_epoca = []
    losses_g_epoca = []

    for batch_imgs, batch_labels in dataset_gan:
        batch_labels = tf.reshape(batch_labels, (-1, 1))
        loss_d, loss_g = train_step(batch_imgs, batch_labels)
        losses_d_epoca.append(float(loss_d))
        losses_g_epoca.append(float(loss_g))

    loss_d_prom = np.mean(losses_d_epoca)
    loss_g_prom = np.mean(losses_g_epoca)
    historial_loss_d.append(loss_d_prom)
    historial_loss_g.append(loss_g_prom)

    print(f'Epoca {epoca}/{EPOCHS} - Loss_D: {loss_d_prom:.4f} - Loss_G: {loss_g_prom:.4f} - Tiempo: {time.time()-t0:.1f}s')

    if epoca == epoca_intermedia:
        imagenes_intermedias = generador([z_fixed, y_fixed], training=False).numpy()

imagenes_final = generador([z_fixed, y_fixed], training=False).numpy()

`Loss_D` pasa de **1,0094** a **1,2362**, mientras `Loss_G` desciende de **1,9586** a **1,1097**. Entre las épocas se observan cambios y oscilaciones, especialmente durante la primera mitad, pero ninguna pérdida colapsa a cero ni crece sin control. Esto indica que las dos redes continuaron compitiendo sin que una dominara completamente.

#### 4.1.4 Curvas de pérdida

graficamos `Loss_D` y `Loss_G` sobre las mismas 30 épocas para evaluar visualmente el equilibrio adversarial.


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(historial_loss_d, label='Loss Discriminador')
plt.plot(historial_loss_g, label='Loss Generador')
plt.xlabel('Epoca')
plt.ylabel('Loss')
plt.title('Curvas de perdida - Entrenamiento cGAN')
plt.legend()
plt.show()

al inicio el Generador tiene la mayor pérdida; después ambas curvas oscilan y se aproximan. En las últimas épocas `Loss_D` se estabiliza cerca de 1,24 y `Loss_G` alrededor de 1,11. El comportamiento no es monotónico, como se espera en una GAN, pero tampoco muestra inestabilidad extrema. La calidad debe confirmarse con las imágenes.

#### 4.1.5 Comparación inicio → etapa intermedia → final

mostramos consecutivamente las tres cuadrículas producidas con las mismas entradas fijas.


In [ ]:
mostrar_grilla(imagenes_inicio, 'Inicio del entrenamiento')
mostrar_grilla(imagenes_intermedias, f'Epoca intermedia ({epoca_intermedia})')
mostrar_grilla(imagenes_final, 'Final del entrenamiento')

la evolución es clara. Al inicio todas las salidas son grises y no contienen objetos. En la época 15 aparecen fondos, contrastes y siluetas generales, aunque todavía son muy borrosas. En la época 30 aumentan la definición y la separación por clase: las categorías asociadas con vehículos y barcos presentan estructuras más horizontales y tonos de cielo o agua, mientras varias clases animales muestran formas orgánicas y fondos naturales.

La correspondencia no es perfecta: algunas imágenes de animales siguen siendo ambiguas debido a la baja resolución y a la similitud entre clases. Sin embargo, el cambio entre las tres etapas demuestra que el Generador sí aprendió una distribución visual y no se limitó a repetir la salida inicial.


### 4.2 Generación condicionada por clase

Ahora generamos seis imágenes nuevas para cada una de las diez clases. Esta cuadrícula permite evaluar simultáneamente:

- **Correspondencia:** si cada fila refleja la etiqueta solicitada.
- **Diversidad:** si diferentes vectores `z` producen variaciones dentro de una misma clase.
- **Mode collapse:** si el modelo repite casi exactamente una salida o ignora la condición.


#### 4.2.1 Múltiples muestras para las diez clases

recorremos las diez etiquetas y generamos seis imágenes con vectores latentes diferentes para cada una. Las filas se identifican con el nombre de la categoría para facilitar la comparación.


In [ ]:
N_MUESTRAS_POR_CLASE = 6

fig, axes = plt.subplots(
    NUM_CLASSES,
    N_MUESTRAS_POR_CLASE,
    figsize=(N_MUESTRAS_POR_CLASE * 1.5, NUM_CLASSES * 1.5),
)

for clase in range(NUM_CLASSES):
    z_muestras = tf.random.normal((N_MUESTRAS_POR_CLASE, Z_DIM))
    y_muestras = tf.constant([[clase]] * N_MUESTRAS_POR_CLASE)
    imgs = generador([z_muestras, y_muestras], training=False).numpy()
    imgs = np.clip((imgs + 1.0) / 2.0, 0, 1)

    for j in range(N_MUESTRAS_POR_CLASE):
        axes[clase, j].imshow(imgs[j])
        axes[clase, j].axis('off')
        if j == 0:
            axes[clase, j].text(
                -0.12,
                0.5,
                CLASS_NAMES[clase],
                transform=axes[clase, j].transAxes,
                fontsize=9,
                fontweight='bold',
                ha='right',
                va='center',
            )

fig.suptitle('Muestras generadas por clase - cGAN', fontsize=13)
plt.subplots_adjust(left=0.13, top=0.96, hspace=0.08, wspace=0.05)
plt.show()


se observan variaciones de color, fondo y forma dentro de cada fila, por lo que no existe un *mode collapse* global evidente. La calidad es desigual: algunas categorías presentan patrones relativamente reconocibles, mientras otras —especialmente animales visualmente parecidos— continúan siendo ambiguas. Aun así, las distribuciones de varias filas son distintas, señal de que la etiqueta influye en la salida.

#### 4.2.2 Prueba controlada: `frog` frente a `ship`

usamos exactamente los mismos seis vectores `z` y cambiamos únicamente la etiqueta entre `frog` y `ship`. Esta prueba aísla el efecto de la condición de clase.


In [ ]:
clase_frog = CLASS_NAMES.index('frog')
clase_ship = CLASS_NAMES.index('ship')

z_comparacion = tf.random.normal((6, Z_DIM))
y_frog = tf.constant([[clase_frog]] * 6)
y_ship = tf.constant([[clase_ship]] * 6)

imgs_frog = generador([z_comparacion, y_frog], training=False).numpy()
imgs_ship = generador([z_comparacion, y_ship], training=False).numpy()

mostrar_grilla(imgs_frog, 'y = frog (mismos vectores z)', etiquetas=['frog'] * 6)
mostrar_grilla(imgs_ship, 'y = ship (mismos vectores z)', etiquetas=['ship'] * 6)

con `y = frog` predominan formas orgánicas, tonos verdes y marrones y composiciones asociadas con fondos naturales. Con `y = ship` aparecen estructuras más horizontales, fondos azulados y separaciones semejantes a cielo y agua. Como el ruido fue idéntico en ambos casos, el cambio visual proviene de la etiqueta.

Dentro de cada grupo existen diferencias entre las seis muestras, por lo que el Generador no produce una única imagen fija. No obstante, algunos resultados siguen siendo borrosos y no todos pueden identificarse sin conocer la etiqueta. Se concluye que el condicionamiento fue aprendido de manera parcial pero observable, sin evidencia fuerte de colapso global.


## 5. Desarrollo del Variational Autoencoder

El VAE sigue el esquema:

`x → Encoder → (μ, log σ²) → z → Decoder → x̂`

A diferencia de la cGAN, no utiliza una competencia entre redes ni etiquetas de clase. El Encoder aprende una distribución para cada imagen y el Decoder intenta reconstruirla a partir de una muestra latente.

### 5.1 Encoder probabilístico y reparameterization trick

El Encoder produce `μ` y `log σ²`. El muestreo se reescribe como:

`z = μ + exp(0,5 · log σ²) ⊙ ε`, con `ε ~ N(0, I)`

La aleatoriedad queda aislada en `ε`, mientras las operaciones que dependen de `μ` y `log σ²` son diferenciables. Así, el gradiente puede atravesar `z` y actualizar el Encoder mediante retropropagación.


#### 5.1.1 Capa de muestreo

definimos una capa personalizada `Sampling` que implementa el *reparameterization trick*. Se selecciona una dimensión latente de **128**, adecuada para conservar información suficiente de las imágenes RGB durante la reconstrucción.


In [ ]:
LATENT_DIM = 128

class Sampling(layers.Layer):
    def call(self, inputs):
        mu, log_var = inputs
        epsilon = tf.random.normal(shape=tf.shape(mu))
        return mu + tf.exp(0.5 * log_var) * epsilon

la capa de muestreo queda integrada como una operación diferenciable de Keras y podrá utilizarse dentro del Encoder.

#### 5.1.2 Construcción del Encoder

aplicamos tres convoluciones con `stride=2` para reducir la resolución **32 → 16 → 8 → 4**. Después aplanamos el tensor y usamos dos capas densas independientes para obtener `μ` y `log σ²`; la capa `Sampling` produce `z`. Finalmente mostramos el resumen y guardamos la forma anterior a `Flatten` para construir el Decoder de manera simétrica.


In [ ]:
def construir_encoder(latent_dim=LATENT_DIM):
    entrada = layers.Input(shape=(32, 32, 3), name='encoder_input')

    x = layers.Conv2D(32, kernel_size=3, strides=2, padding='same')(entrada)
    x = layers.LeakyReLU(0.2)(x)

    x = layers.Conv2D(64, kernel_size=3, strides=2, padding='same')(x)
    x = layers.LeakyReLU(0.2)(x)

    x = layers.Conv2D(128, kernel_size=3, strides=2, padding='same')(x)
    x = layers.LeakyReLU(0.2)(x)

    forma_antes_flatten = K.int_shape(x)[1:]
    x = layers.Flatten()(x)

    mu = layers.Dense(latent_dim, name='mu')(x)
    log_var = layers.Dense(latent_dim, name='log_var')(x)
    z = Sampling(name='z')([mu, log_var])

    modelo = Model(entrada, [mu, log_var, z], name='Encoder')
    return modelo, forma_antes_flatten

encoder, forma_pre_flatten = construir_encoder()
encoder.summary()
print('Forma antes de aplanar (la necesitaremos en el Decoder):', forma_pre_flatten)

el Encoder contiene **617.792 parámetros entrenables**. Antes de aplanar, la representación tiene forma **4 × 4 × 128**, que resume la información espacial de la imagen.

#### 5.1.3 Verificación de dimensiones

procesamos cuatro imágenes normalizadas y verificamos las formas de `μ`, `log_var` y `z`.


In [ ]:
x_prueba_vae = x_train_norm[:4]
mu_prueba, log_var_prueba, z_prueba_vae = encoder(x_prueba_vae)

print('Forma de mu:', mu_prueba.shape)
print('Forma de log_var:', log_var_prueba.shape)
print('Forma de z (muestreado):', z_prueba_vae.shape)

 para cuatro imágenes se obtienen tres tensores de forma **(4, 128)**. Esto confirma que el Encoder genera los dos parámetros probabilísticos y una muestra latente de la dimensión prevista.


### 5.2 Decoder

El Decoder transforma `z` en una reconstrucción de 32 × 32 × 3. Primero proyecta el vector a **4 × 4 × 128** y después utiliza convoluciones transpuestas para recuperar progresivamente la resolución. La capa final usa `tanh`, compatible con la normalización `[-1, 1]`.


#### 5.2.1 Construcción del Decoder

proyectamos el vector latente a la forma espacial guardada por el Encoder y aplicamos tres capas `Conv2DTranspose` para expandir **4 → 8 → 16 → 32**. La última convolución produce los tres canales RGB.


In [ ]:
def construir_decoder(latent_dim=LATENT_DIM, forma_pre_flatten=forma_pre_flatten):
    entrada_z = layers.Input(shape=(latent_dim,), name='decoder_input')

    unidades = int(np.prod(forma_pre_flatten))
    x = layers.Dense(unidades)(entrada_z)
    x = layers.LeakyReLU(0.2)(x)
    x = layers.Reshape(forma_pre_flatten)(x)

    x = layers.Conv2DTranspose(128, kernel_size=3, strides=2, padding='same')(x)
    x = layers.LeakyReLU(0.2)(x)

    x = layers.Conv2DTranspose(64, kernel_size=3, strides=2, padding='same')(x)
    x = layers.LeakyReLU(0.2)(x)

    x = layers.Conv2DTranspose(32, kernel_size=3, strides=2, padding='same')(x)
    x = layers.LeakyReLU(0.2)(x)

    salida = layers.Conv2D(3, kernel_size=3, padding='same', activation='tanh', name='reconstructed_image')(x)

    modelo = Model(entrada_z, salida, name='Decoder')
    return modelo

decoder = construir_decoder()
decoder.summary()

el Decoder tiene **504.899 parámetros entrenables** y termina correctamente en una salida `(32, 32, 3)`.

#### 5.2.2 Verificación de dimensiones

decodificamos los cuatro vectores latentes de prueba y comprobamos la forma y el rango de las reconstrucciones.


In [ ]:
recon_prueba = decoder(z_prueba_vae)
print('Forma de la reconstruccion:', recon_prueba.shape)
print('Rango de valores reconstruidos: [{:.3f}, {:.3f}]'.format(
    float(tf.reduce_min(recon_prueba)), float(tf.reduce_max(recon_prueba))))

el Decoder produce un lote de forma **(4, 32, 32, 3)**. Antes del entrenamiento los valores están cerca de cero, pero permanecen dentro de `[-1, 1]`, como corresponde a la activación `tanh`.


### 5.3 Función de pérdida

El VAE optimiza:

`L_VAE = L_rec + β · L_KL`

- **`L_rec`:** suma del error cuadrático sobre los píxeles; obliga al modelo a conservar la información necesaria para reconstruir la imagen.
- **`L_KL`:** divergencia de Kullback-Leibler frente a `N(0, I)`; organiza el espacio latente y facilita el muestreo de imágenes nuevas.
- **`β`:** controla el equilibrio. Un valor excesivo puede producir reconstrucciones genéricas; uno demasiado pequeño puede dejar un espacio latente irregular.

Se utiliza **β = 1,0**, correspondiente a un VAE estándar, y se registran por separado los tres términos para analizar su interacción.


#### 5.3.1 Modelo VAE y pérdidas separadas

 creamos una subclase de `Model` con un `train_step` personalizado. En cada lote calculamos la reconstrucción, `L_rec`, `L_KL` y `L_total`, aplicamos los gradientes y actualizamos tres métricas independientes.


In [ ]:
BETA = 1.0

class VAE(Model):
    def __init__(self, encoder, decoder, beta=BETA, **kwargs):
        super().__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder
        self.beta = beta

        self.loss_total_tracker = keras.metrics.Mean(name='loss_total')
        self.loss_rec_tracker = keras.metrics.Mean(name='loss_rec')
        self.loss_kl_tracker = keras.metrics.Mean(name='loss_kl')

    @property
    def metrics(self):
        return [self.loss_total_tracker, self.loss_rec_tracker, self.loss_kl_tracker]

    def train_step(self, data):
        with tf.GradientTape() as tape:
            mu, log_var, z = self.encoder(data, training=True)
            reconstruccion = self.decoder(z, training=True)

            loss_rec = tf.reduce_mean(
                tf.reduce_sum(tf.square(data - reconstruccion), axis=[1, 2, 3])
            )
            loss_kl = -0.5 * tf.reduce_mean(
                tf.reduce_sum(1 + log_var - tf.square(mu) - tf.exp(log_var), axis=1)
            )
            loss_total = loss_rec + self.beta * loss_kl

        grads = tape.gradient(loss_total, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))

        self.loss_total_tracker.update_state(loss_total)
        self.loss_rec_tracker.update_state(loss_rec)
        self.loss_kl_tracker.update_state(loss_kl)

        return {
            'loss_total': self.loss_total_tracker.result(),
            'loss_rec': self.loss_rec_tracker.result(),
            'loss_kl': self.loss_kl_tracker.result(),
        }

    def call(self, data):
        mu, log_var, z = self.encoder(data)
        return self.decoder(z)

el modelo queda preparado para entrenar Encoder y Decoder de manera conjunta y conservar el historial separado de las pérdidas.

#### 5.3.2 Entrenamiento del VAE

compilamos el VAE con Adam, tasa de aprendizaje `1e-3`, y lo entrenamos durante 30 épocas con lotes de 128 imágenes.


In [ ]:
vae = VAE(encoder, decoder, beta=BETA)
vae.compile(optimizer=Adam(learning_rate=1e-3))

EPOCHS_VAE = 30

historial_vae = vae.fit(
    x_train_norm,
    epochs=EPOCHS_VAE,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

la pérdida total disminuye de **323,38** a **180,69**. La reconstrucción baja de **277,68** a **114,53**, mientras la KL aumenta de **45,70** a **66,16** y luego se estabiliza. Esto muestra el compromiso esperado: el modelo mejora la reconstrucción mientras conserva una regularización latente activa. Como `L_KL` no cae a cero, no se observa evidencia de *posterior collapse* en este entrenamiento.

#### 5.3.3 Curvas de las tres pérdidas

 graficamos `L_total`, `L_rec` y `L_KL` por separado para examinar su evolución durante las 30 épocas.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(historial_vae.history['loss_total'])
axes[0].set_title('Loss total')
axes[0].set_xlabel('Epoca')

axes[1].plot(historial_vae.history['loss_rec'], color='orange')
axes[1].set_title('Loss de reconstruccion')
axes[1].set_xlabel('Epoca')

axes[2].plot(historial_vae.history['loss_kl'], color='green')
axes[2].set_title('Loss KL')
axes[2].set_xlabel('Epoca')

plt.tight_layout()
plt.show()

 `L_total` y `L_rec` caen con rapidez en las primeras épocas y después mejoran de manera más gradual. `L_KL` aumenta al comienzo y se estabiliza alrededor de 66, señal de que el Encoder utiliza el espacio latente en lugar de ignorarlo. Las curvas son suaves y no presentan oscilaciones fuertes, por lo que el entrenamiento del VAE fue más estable que el de la cGAN.


## 6. Análisis del espacio latente y resultados del VAE

### 6.1 Reconstrucción de imágenes de prueba

seleccionamos una imagen de prueba de ocho clases diferentes. Estas imágenes no participaron en el entrenamiento. Las codificamos y decodificamos para comparar cada original con su reconstrucción y evaluar la capacidad de generalización.


 buscamos un ejemplo aleatorio de cada una de las primeras ocho clases, obtenemos su representación latente con el Encoder, reconstruimos con el Decoder y mostramos originales y resultados en dos filas.


In [ ]:
N_CLASES_MOSTRAR = 8
indices_muestra = []

for clase in range(N_CLASES_MOSTRAR):
    idx_clase = np.where(y_test == clase)[0]
    indices_muestra.append(np.random.choice(idx_clase))

x_originales = x_test_norm[indices_muestra]
mu_test, log_var_test, z_test = encoder(x_originales, training=False)
x_reconstruidas = decoder(z_test, training=False).numpy()

fig, axes = plt.subplots(2, N_CLASES_MOSTRAR, figsize=(N_CLASES_MOSTRAR * 1.6, 4))

for i in range(N_CLASES_MOSTRAR):
    img_orig = np.clip((x_originales[i] + 1.0) / 2.0, 0, 1)
    img_recon = np.clip((x_reconstruidas[i] + 1.0) / 2.0, 0, 1)

    axes[0, i].imshow(img_orig)
    axes[0, i].set_title(CLASS_NAMES[y_test[indices_muestra[i]]], fontsize=8)
    axes[0, i].axis('off')

    axes[1, i].imshow(img_recon)
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Original', fontsize=10)
axes[1, 0].set_ylabel('Reconstruccion', fontsize=10)

plt.tight_layout()
plt.show()

 el VAE conserva el color dominante, la posición aproximada, el fondo y la silueta general. Esto se aprecia con claridad en el automóvil rojo, el avión sobre fondo azul y el caballo sobre un entorno verde. Sin embargo, las reconstrucciones pierden bordes, texturas y detalles finos; las categorías animales más complejas resultan especialmente borrosas.

El comportamiento es coherente con una pérdida MSE: ante varias reconstrucciones posibles, el modelo favorece promedios suaves que reducen el error numérico. Por tanto, el VAE generaliza y reconstruye la estructura global, pero no recupera completamente la nitidez de CIFAR-10.


### 6.2 Generación desde el espacio latente

Para evaluar la capacidad generativa real, muestreamos `z ~ N(0, I)` sin utilizar imágenes de entrada y calculamos `x_nueva = Decoder(z)`. Si la regularización KL organizó adecuadamente el espacio, estas muestras deben producir estructuras visuales coherentes.


 generamos 16 vectores normales independientes, los decodificamos y presentamos las imágenes nuevas en una cuadrícula de 4 × 4.


In [ ]:
N_MUESTRAS_VAE = 16

z_random = tf.random.normal((N_MUESTRAS_VAE, LATENT_DIM))
imagenes_nuevas = decoder(z_random, training=False).numpy()
imagenes_nuevas = np.clip((imagenes_nuevas + 1.0) / 2.0, 0, 1)

fig, axes = plt.subplots(4, 4, figsize=(8, 8))
for i, ax in enumerate(axes.flat):
    ax.imshow(imagenes_nuevas[i])
    ax.axis('off')

fig.suptitle('Imagenes generadas por el VAE desde z ~ N(0, I)')
plt.tight_layout()
plt.show()

 las 16 muestras presentan diversidad de colores, fondos y composiciones, por lo que el Decoder no genera una única salida. Varias imágenes muestran una estructura central y separación entre objeto y fondo, pero permanecen borrosas y algunas mezclan rasgos de diferentes categorías.

A diferencia de la cGAN, el VAE utilizado no recibe etiquetas; por ello no se puede solicitar una clase concreta y algunas muestras son semánticamente ambiguas. Visualmente, la cGAN ofrece mayor contraste y control de clase, mientras el VAE produce transiciones más suaves y resultados menos definidos.


### 6.3 Interpolación latente

Seleccionamos una imagen de *cat* y otra de *dog*, obtenemos sus medias latentes `μ_A` y `μ_B` y construimos diez puntos:

`z(α) = (1 - α) μ_A + α μ_B`, con `α ∈ [0, 1]`

Usamos la media, y no una muestra aleatoria, para que la trayectoria sea determinista.


 calculamos diez valores de `α`, interpolamos linealmente entre las dos representaciones, decodificamos cada punto y mostramos la secuencia completa.


In [ ]:
N_PASOS = 10

idx_a = np.where(y_test == CLASS_NAMES.index('cat'))[0][0]
idx_b = np.where(y_test == CLASS_NAMES.index('dog'))[0][0]

x_a = x_test_norm[idx_a:idx_a+1]
x_b = x_test_norm[idx_b:idx_b+1]

mu_a, _, _ = encoder(x_a, training=False)
mu_b, _, _ = encoder(x_b, training=False)

alphas = np.linspace(0, 1, N_PASOS)
z_interpolados = np.array([(1 - a) * mu_a.numpy() + a * mu_b.numpy() for a in alphas]).squeeze(1)

imagenes_interpoladas = decoder(z_interpolados, training=False).numpy()
imagenes_interpoladas = np.clip((imagenes_interpoladas + 1.0) / 2.0, 0, 1)

fig, axes = plt.subplots(1, N_PASOS, figsize=(N_PASOS * 1.5, 2))
for i, ax in enumerate(axes):
    ax.imshow(imagenes_interpoladas[i])
    ax.set_title(f'α={alphas[i]:.1f}', fontsize=8)
    ax.axis('off')

fig.suptitle('Interpolacion en el espacio latente: cat → dog')
plt.tight_layout()
plt.show()

 la secuencia cambia gradualmente de color, fondo y forma; no aparecen saltos abruptos ni cuadros de ruido entre los extremos. Las imágenes centrales combinan rasgos de ambas representaciones, aunque conservan la borrosidad propia del VAE.

Este resultado respalda que la regularización KL produjo una región latente continua entre los dos ejemplos. La interpolación no demuestra que la totalidad del espacio sea perfecta, pero sí evidencia que esta trayectoria concreta puede recorrerse de manera suave y decodificable.


## 7. Comparación experimental entre cGAN y VAE

La comparación combina las imágenes, las curvas de entrenamiento y un indicador descriptivo de diversidad. Para este último se calcula la desviación estándar promedio de los píxeles entre muestras. Un valor mayor indica mayor variación numérica, aunque no sustituye una métrica perceptual como FID o KID ni garantiza diversidad semántica.


### 7.1 Diversidad aproximada de la cGAN

 generamos 20 imágenes por clase, calculamos la dispersión promedio de los píxeles dentro de cada categoría y obtenemos un promedio global.


In [ ]:
def diversidad_promedio(imagenes):
    imagenes = imagenes.reshape(imagenes.shape[0], -1)
    return float(np.mean(np.std(imagenes, axis=0)))

diversidades_cgan = []
for clase in range(NUM_CLASSES):
    z_muestras = tf.random.normal((20, Z_DIM))
    y_muestras = tf.constant([[clase]] * 20)
    imgs = generador([z_muestras, y_muestras], training=False).numpy()
    diversidades_cgan.append(diversidad_promedio(imgs))

diversidad_cgan_prom = np.mean(diversidades_cgan)
print('Diversidad promedio por clase (cGAN):', np.round(diversidades_cgan, 4))
print('Diversidad promedio global (cGAN):', round(diversidad_cgan_prom, 4))

 la diversidad por clase varía entre **0,3432** y **0,5141**, con promedio global **0,4472**. Las diferencias entre categorías confirman que el modelo representa algunas clases con mayor variación que otras. El valor, junto con la cuadrícula visual, no muestra repetición idéntica generalizada.

### 7.2 Diversidad aproximada del VAE

 generamos 200 imágenes desde `N(0, I)` y calculamos el mismo indicador para comparar ambos modelos en una escala común.


In [ ]:
z_random_vae = tf.random.normal((200, LATENT_DIM))
imgs_vae = decoder(z_random_vae, training=False).numpy()
diversidad_vae_prom = diversidad_promedio(imgs_vae)

print('Diversidad promedio global (VAE):', round(diversidad_vae_prom, 4))
print('\nComparacion directa de diversidad:')
print(f'  cGAN: {diversidad_cgan_prom:.4f}')
print(f'  VAE : {diversidad_vae_prom:.4f}')

 el VAE obtiene **0,4032**, frente a **0,4472** de la cGAN. Bajo este indicador, la cGAN produjo mayor variación de píxeles. La diferencia es moderada y debe interpretarse junto con las imágenes: una dispersión alta también puede incluir cambios de fondo o artefactos, no únicamente diversidad semántica.

### 7.3 Comparación de las curvas de entrenamiento

 colocamos lado a lado las pérdidas adversariales de la cGAN y los tres componentes de pérdida del VAE.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(historial_loss_d, label='Loss_D')
axes[0].plot(historial_loss_g, label='Loss_G')
axes[0].set_title('cGAN - Entrenamiento adversarial')
axes[0].set_xlabel('Epoca')
axes[0].legend()

axes[1].plot(historial_vae.history['loss_total'], label='Loss total')
axes[1].plot(historial_vae.history['loss_rec'], label='Loss reconstruccion')
axes[1].plot(historial_vae.history['loss_kl'], label='Loss KL')
axes[1].set_title('VAE - Optimizacion directa')
axes[1].set_xlabel('Epoca')
axes[1].legend()

plt.tight_layout()
plt.show()

la cGAN presenta oscilaciones asociadas al equilibrio entre dos redes; sus pérdidas no pueden interpretarse como una optimización convencional. El VAE muestra una disminución suave y sostenida de `L_total` y `L_rec`, mientras `L_KL` crece al inicio y se estabiliza. Por tanto, el VAE fue más estable y fácil de diagnosticar, mientras la cGAN exigió observar continuamente pérdidas e imágenes.



### 7.5 Síntesis de la comparación

Los resultados muestran un compromiso claro. La cGAN ofrece control por etiqueta, mayor contraste y una diversidad numérica ligeramente superior, pero su entrenamiento es más difícil de interpretar y la calidad entre clases es irregular. El VAE presenta un entrenamiento estable, reconstruye imágenes y permite interpolar suavemente, aunque genera resultados más borrosos y no controla la clase en esta versión.

Por tanto, ninguno es universalmente mejor: la cGAN es más apropiada cuando se prioriza el control condicional y la nitidez relativa; el VAE resulta más útil cuando se necesita reconstrucción, continuidad e interpretación del espacio latente.


## 8. Conclusiones

1. **La adaptación a CIFAR-10 funcionó correctamente:** ambas arquitecturas reciben y producen tensores de 32 × 32 × 3, utilizan el rango `[-1, 1]` y generan imágenes RGB sin depender de archivos externos.

2. **La cGAN aprendió información visual y condicional:** las salidas pasaron de imágenes grises sin estructura a composiciones con colores, fondos y siluetas diferenciadas. La prueba con el mismo `z` para `frog` y `ship` confirmó que cambiar la etiqueta modifica la distribución generada.

3. **El condicionamiento todavía es imperfecto:** después de 30 épocas varias imágenes, especialmente de clases animales, siguen siendo ambiguas. Un segundo experimento debería aumentar las épocas y evaluar arquitecturas más robustas, por ejemplo normalización espectral o bloques residuales.

4. **No se observó un *mode collapse* global fuerte:** dentro de cada clase hubo cambios de color, fondo y forma. Además, la diversidad aproximada de la cGAN fue **0,4472**, superior al **0,4032** del VAE, aunque esta métrica de píxeles no reemplaza una evaluación perceptual.

5. **La cGAN fue el modelo más difícil de entrenar e interpretar:** `Loss_D` y `Loss_G` oscilaron por la competencia adversarial y fue necesario revisar simultáneamente curvas e imágenes. Aun así, ninguna red dominó completamente al final del entrenamiento.

6. **El VAE aprendió reconstrucciones útiles pero borrosas:** la pérdida total bajó de **323,38** a **180,69** y las reconstrucciones conservaron color, posición y estructura general, pero perdieron bordes y texturas finas debido al uso de MSE.

7. **El espacio latente del VAE mostró continuidad:** la interpolación de diez pasos entre *cat* y *dog* produjo cambios suaves y sin saltos a ruido. La KL se mantuvo activa y se estabilizó, por lo que no hubo evidencia de *posterior collapse*.

8. **Los modelos ofrecen ventajas diferentes:** la cGAN fue mejor para controlar la clase y lograr mayor contraste; el VAE fue mejor para reconstruir, interpolar y analizar el espacio latente. Como trabajo futuro se propone incorporar FID/KID, precisión de un clasificador sobre imágenes generadas, un VAE condicional y pérdidas perceptuales para obtener una comparación más completa.
